# Tutorial: Multiple Myeloma Datasets
## Notebook 04 — Synthetic Data Augmentation

This notebook builds a pipeline for **synthetic bone marrow smear images** to augment the plasma cell detection training set. It trains a generative model on real plasma cells and composes the generated cells onto synthetic background templates to form full smears.

**What you will learn:**
- How to estimate a slide's background light and other per-slide statistics
- How to model those statistics with KDE to generate synthetic background templates
- How to crop plasma cells from bone marrow smear images and isolate each one with its segmentation mask
- How to transform crops into a new color space (color deconvolution), using each slide's background light
- How to train a generative (diffusion) model on those maps to synthesize new cells
- How to compose the generated synthetic cells onto templates to create new synthetic smear images
- How to check whether this synthetic augmentation improves a detector on held-out test data

**Pipeline steps:**
1. Compute per-slide statistics (background light, cell count, size, position) — demonstrated on a few sample slides
2. Load `slides.csv` and `cell_boxes.csv` (the same statistics, precomputed for the full dataset)
3. Model per-slide statistics with KDE to generate background templates
4. Crop and mask the plasma cells from the original slides
5. Transform crops into a new color space (color deconvolution), using each slide's background light
6. Load the concentration maps precomputed for the full dataset
7. Train a generative (diffusion) model on those maps
8. Sample new synthetic cells from the trained model
9. Compose the generated cells onto the templates, producing full synthetic smear images
10. Compare a baseline detector against one trained with the synthetic augmentation

---
## 0. Environment Setup

In [ ]:
!pip install --upgrade pip
!pip install numpy pandas matplotlib seaborn pillow scipy scikit-learn tifffile torch diffusers opencv-python-headless ultralytics huggingface_hub

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.cluster import MiniBatchKMeans
from IPython.display import display

sns.set_theme()

SLIDES_DIR = Path('data/slides')
TRAIN_DIR = SLIDES_DIR / 'train'
CELLS_DIR = Path('data/plasma_cells')

print('Environment configured.')

### Download the dataset

Everything this notebook reads — the slides, the plasma cell crops/masks/stains, and the pretrained models — is packaged as `data.zip` on the Hugging Face Hub ([`LABIA-UFBA/myeloma_aug`](https://huggingface.co/datasets/LABIA-UFBA/myeloma_aug)). We download and extract it into `data/`, matching the paths defined above.

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

# The slides, plasma cell crops/masks/stains and pretrained models are packaged as
# data.zip on the Hugging Face Hub. Download and extract it into data/ (once).
if not Path('data').exists():
    print('Downloading data.zip from Hugging Face...')
    zip_path = hf_hub_download(repo_id='LABIA-UFBA/myeloma_aug',
                               filename='data.zip', repo_type='dataset')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('.')
    print('Extracted to data/')
else:
    print('data/ already present.')

---
## Part 1 — Per-Slide Statistics

A **background template** is a synthetic slide layout — a background color plus cell positions and sizes — sampled to match real slides statistically without copying any. Part 3 fits KDE models over per-slide statistics to generate these templates; generated cells (Parts 4–8) are pasted onto the sampled positions (Part 9).

To fit those KDEs, we characterize each real slide by:

- **Background light** — the slide's background color, used later to normalize the color space.
- **Cell count, size and position** — how many plasma cells, and their layout.

We compute each on a few sample slides, then load the precomputed values for the full dataset.

### 1.1 Background Light Estimation

Each slide has its own background light — the color of its unstained regions — which varies with staining intensity and illumination at capture time. We estimate it by downsampling the slide to a thumbnail, clustering its pixel colors with K-Means (k=5), and picking the **brightest centroid**, since background pixels are assumed to be brighter than cells and debris.

Below we demonstrate this on three sample slides with visibly different background colors.

In [ ]:
def estimate_bg_light(image, k=5, max_size=128):
    """Estimate the background light of a slide via K-Means clustering."""
    # Downsample first: color statistics survive resizing, clustering doesn't need full resolution.
    thumb = image.copy()
    thumb.thumbnail((max_size, max_size))
    pixels = np.array(thumb).reshape(-1, 3)

    # Background pixels are the brightest cluster; cells and debris pull the others down.
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=0).fit(pixels)
    brightness = kmeans.cluster_centers_.mean(axis=1)
    centroid = kmeans.cluster_centers_[np.argmax(brightness)]
    return centroid.round().astype(int)

In [ ]:
# Three sample slides with visibly different background colors
sample_paths = [TRAIN_DIR / 'images' / name for name in
                ['IMG_0538.jpg', 'P_20221014_145236.jpg', 'IMG01144.jpg']]

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(6, 3 * len(sample_paths)),
                          gridspec_kw={'width_ratios': [3, 1]})
fig.suptitle('Background Light Estimation (K-Means, k=5)')

for i, img_path in enumerate(sample_paths):
    img = Image.open(img_path).convert('RGB')
    bg_light = estimate_bg_light(img)
    rgb = tuple(int(v) for v in bg_light)

    ax_img, ax_color = axes[i]
    ax_img.imshow(img)
    ax_img.set_title(img_path.name, fontsize=9)
    ax_img.axis('off')

    # Solid swatch of the estimated color, for visual comparison against the slide
    ax_color.imshow([[bg_light]])
    ax_color.set_title(f'RGB: {rgb}', fontsize=9)
    ax_color.axis('off')

plt.tight_layout()
plt.show()

### 1.2 Cell Count, Size and Position

Besides the background light, a slide is characterized by its plasma cells: how many, and each one's position and size. This comes from the YOLO label file shipped with every image — one box per line, `class_id x_center y_center width height`, where the four box coordinates are normalized to 0–1 and `class_id` is the integer class index (`0`, `1`, …).

Steps, per slide:

1. **Read the label file** — the `.txt` with the same name as the image.
2. **Keep only plasma cells.** The dataset has two classes, `0 = plasma_cell` and `1 = non_plasma_cell`. This pipeline focuses on plasma cells, so we keep class `0` and drop the rest here.
3. **Count them** — the number of kept boxes, one value per slide (stored with the background light from 1.1).
4. **Record position and size.** For each kept box, its center `(x, y)` and size `(w, h)`, one row per cell. Slides have different aspect ratios, so we first pad each image to a square (centered on the shorter side) and express the coordinates relative to that square — a given `x` or `w` then means the same position/size on any slide.

This yields statistics at two levels: one row per slide (count + background light) and one row per cell (position + size). Below we run it on the sample slides, draw the boxes back to check them, and show the values.

In [ ]:
def load_plasma_cell_boxes(img_path):
    """Read plasma_cell (class 0) boxes, normalized against the image padded to a square."""
    img_width, img_height = Image.open(img_path).size
    size = max(img_width, img_height)
    pad_x, pad_y = (size - img_width) / 2, (size - img_height) / 2

    label_path = TRAIN_DIR / 'labels' / f'{img_path.stem}.txt'
    rows = []
    for class_id, x, y, w, h in np.loadtxt(label_path, ndmin=2):
        if int(class_id) != 0:
            continue
        rows.append({
            'slide': img_path.stem,
            'x': (x * img_width + pad_x) / size,
            'y': (y * img_height + pad_y) / size,
            'w': (w * img_width) / size,
            'h': (h * img_height) / size,
        })
    return pd.DataFrame(rows, columns=['slide', 'x', 'y', 'w', 'h'])


def pad_to_square(image):
    """Pad an image to a centered square canvas (gray padding, so it stays visible)."""
    size = max(image.size)
    canvas = Image.new('RGB', (size, size), (180, 180, 180))
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas

In [ ]:
slide_stats, all_boxes = [], []

fig, axes = plt.subplots(1, len(sample_paths), figsize=(5 * len(sample_paths), 5))

for ax, img_path in zip(axes, sample_paths):
    img = Image.open(img_path).convert('RGB')
    square = pad_to_square(img)
    size = square.width

    boxes = load_plasma_cell_boxes(img_path)
    all_boxes.append(boxes)

    bg = estimate_bg_light(img)
    slide_stats.append({'slide': img_path.stem, 'n_plasma_cells': len(boxes),
                        'bg_r': bg[0], 'bg_g': bg[1], 'bg_b': bg[2]})

    ax.imshow(square)
    for _, box in boxes.iterrows():
        # Boxes are normalized against the padded square, so we scale straight back up
        ax.add_patch(plt.Rectangle(((box.x - box.w / 2) * size, (box.y - box.h / 2) * size),
                                    box.w * size, box.h * size,
                                    fill=False, edgecolor='crimson', linewidth=1.5))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

# One row per slide (count + background light), one row per cell (position + size)
display(pd.DataFrame(slide_stats))
display(pd.concat(all_boxes, ignore_index=True))

---
## Part 2 — Load the Precomputed Statistics

The same statistics are precomputed for all 512 training slides and saved as two CSVs:

- **`slides.csv`** — one row per slide: background light (`bg_r`, `bg_g`, `bg_b`), plasma cell count (`n_plasma_cells`), image dimensions.
- **`cell_boxes.csv`** — one row per plasma cell: position and size (`x`, `y`, `w`, `h`) on the padded square.

Both were built with the steps from Part 1. From here we work from these tables.

In [ ]:
slides = pd.read_csv(TRAIN_DIR / 'slides.csv')
cell_boxes = pd.read_csv(TRAIN_DIR / 'cell_boxes.csv')

print(f'slides.csv: {len(slides)} slides')
print(f'cell_boxes.csv: {len(cell_boxes)} plasma cells')

display(slides.head())
display(cell_boxes.head())

---
## Part 3 — Model the Statistics with KDE

A Kernel Density Estimate (KDE) fits a smooth density to a set of observations; sampling from it gives new values with the same distribution as the real data, without copying any slide. We fit one KDE per quantity that defines a template:

- **counts** — plasma cells per slide (from `slides.n_plasma_cells`)
- **background** — background light color, jointly over R, G, B (from `slides`)
- **coords** — cell position on the square, jointly over `(x, y)` (from `cell_boxes`)
- **sizes** — cell scale, taken as `max(w, h)` (from `cell_boxes`)

Each quantity is bounded — counts ≥ 1, colors 0–255, coords and sizes 0–1 — so we wrap the KDE in a helper that rejects out-of-bounds samples.

In [ ]:
from scipy.stats import gaussian_kde


class BoundedKDE:
    """Gaussian KDE restricted to [low, high]: samples outside the bounds are rejected."""
    def __init__(self, data, low=-np.inf, high=np.inf):
        self.kde = gaussian_kde(data)
        self.low, self.high = low, high

    def sample(self, n, rng):
        kept = np.empty((0, self.kde.dataset.shape[0]))
        while len(kept) < n:
            s = self.kde.resample(n, seed=rng).T
            s = s[((s >= self.low) & (s <= self.high)).all(axis=1)]
            kept = np.vstack([kept, s])
        return kept[:n]


# max(w, h): one scale per cell
sizes = cell_boxes[['w', 'h']].max(axis=1).values

samplers = {
    'counts': BoundedKDE(slides['n_plasma_cells'].values.reshape(1, -1), low=1),
    'background': BoundedKDE(slides[['bg_r', 'bg_g', 'bg_b']].values.T, low=0, high=255),
    'coords': BoundedKDE(cell_boxes[['x', 'y']].values.T, low=0, high=1),
    'sizes': BoundedKDE(sizes.reshape(1, -1), low=0, high=1),
}

print('Fitted KDEs:', list(samplers))

With the four densities fitted, a template is: draw a background color, a cell count, then that many positions and sizes. Boxes are placed one at a time, skipping any that would overlap one already placed. The function below returns a background color plus non-overlapping `(x, y, size)` boxes on the unit square; we draw a few to check.

In [ ]:
def overlaps(box, others):
    """True if the (x, y, size) box overlaps any box already placed."""
    x, y, s = box
    return any(abs(x - ox) < (s + o) / 2 and abs(y - oy) < (s + o) / 2
               for ox, oy, o in others)


def sample_template(rng):
    """Sample one template: a background color and non-overlapping (x, y, size) boxes."""
    bg = samplers['background'].sample(1, rng)[0].round().astype(int)
    n = int(round(samplers['counts'].sample(1, rng)[0, 0]))

    boxes = []
    while len(boxes) < n:
        x, y = samplers['coords'].sample(1, rng)[0]
        s = samplers['sizes'].sample(1, rng)[0, 0]
        if not overlaps((x, y, s), boxes):
            boxes.append((x, y, s))
    return bg, boxes


rng = np.random.default_rng(0)  # explicit generator so the templates are reproducible

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax in axes:
    bg, boxes = sample_template(rng)

    ax.set_facecolor(tuple(bg / 255))
    ax.grid(False)
    for x, y, s in boxes:
        ax.add_patch(plt.Rectangle((x - s / 2, y - s / 2), s, s,
                                    fill=False, edgecolor='crimson', linewidth=1.5))
    ax.set_title(f'{len(boxes)} cells   RGB {tuple(int(v) for v in bg)}', fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)  # origin at top-left, like an image
    ax.set_aspect('equal')
    ax.set_xticks([]), ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## Part 4 — Crop and Mask the Plasma Cells

A plasma cell **crop** is the patch of slide pixels inside one of the bounding boxes from 1.2. We demonstrate it on a single box from a sample slide, cropping from the padded square so the coordinates match those computed in 1.2.

In [ ]:
# Demonstrate a crop: cut one plasma-cell box out of its slide
img_path = sample_paths[0]
square = pad_to_square(Image.open(img_path).convert('RGB'))
size = square.width

box = load_plasma_cell_boxes(img_path).iloc[0]
left, top = (box.x - box.w / 2) * size, (box.y - box.h / 2) * size
crop = square.crop((left, top, left + box.w * size, top + box.h * size))

fig, (ax_slide, ax_crop) = plt.subplots(1, 2, figsize=(7, 4))
ax_slide.imshow(square)
ax_slide.add_patch(plt.Rectangle((left, top), box.w * size, box.h * size,
                                  fill=False, edgecolor='crimson', linewidth=1.5))
ax_slide.set_title(img_path.name, fontsize=9)
ax_crop.imshow(crop)
ax_crop.set_title('extracted crop', fontsize=9)
for ax in (ax_slide, ax_crop):
    ax.axis('off')
plt.tight_layout()
plt.show()

The full set of crops is precomputed in `data/plasma_cells/crops/`, one per cell (`<slide>_<i>.jpg`). `data/plasma_cells/masks/` holds a binary segmentation mask per crop, same name (`<slide>_<i>.png`), provided with the dataset: 255 on the cell, 0 on the background.

For one slide, per cell, we show:

1. **crop** — the cutout
2. **mask** — the binary segmentation
3. **cell** — crop with the background removed via the mask

The masked cell is the input to Part 5 (color deconvolution), whose 2-channel output trains the generative model.

In [ ]:
def load_mask(mask_path):
    """Load a binary PNG mask into a boolean array."""
    return np.array(Image.open(mask_path)) > 0

In [ ]:
demo_stem = 'IMG01117'
crop_paths = sorted((CELLS_DIR / 'crops').glob(f'{demo_stem}_*.jpg'))

fig, axes = plt.subplots(len(crop_paths), 3, figsize=(5, 1.8 * len(crop_paths)))
axes = axes.reshape(len(crop_paths), 3)
fig.suptitle(f'Crops and masks — {demo_stem}')

for (ax_crop, ax_mask, ax_cell), crop_path in zip(axes, crop_paths):
    crop = Image.open(crop_path).convert('RGB')
    mask = load_mask(CELLS_DIR / 'masks' / f'{crop_path.stem}.png')

    cell = np.array(crop).copy()
    cell[~mask] = 255  # drop the background

    ax_crop.imshow(crop)
    ax_mask.imshow(mask, cmap='gray')
    ax_cell.imshow(cell)
    for ax in (ax_crop, ax_mask, ax_cell):
        ax.axis('off')

for ax, title in zip(axes[0], ['crop', 'mask', 'cell (bg removed)']):
    ax.set_title(title, fontsize=9)

plt.tight_layout()
plt.show()

---
## Part 5 — Color Deconvolution

We represent a stained cell by dye concentration instead of raw RGB. Under transmitted light, the Beer-Lambert law makes **optical density** linear in dye concentration:

$$\text{OD} = -\log_{10}\!\left(\frac{I}{I_0}\right)$$

$I$ is the pixel color, $I_0$ the background light (the per-slide value from Part 1, in `slides.csv`). Each dye is a unit RGB absorption vector; stacking the two Wright-Giemsa dyes (**Azure B**, **Eosin Y**) gives a 2×3 **stain matrix** `S`.

Since $\text{OD} = C \cdot S$, the two concentrations `C` are $\text{OD}\cdot S^{+}$ (least-squares via the pseudo-inverse), clipped at zero — mapping each 3-channel crop to a **2-channel concentration map**. Below we deconvolve the masked cells from Part 4, using each cell's source-slide background light as $I_0$.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Wright-Giemsa stain matrix S: unit RGB absorption vectors, one row per dye
S = np.array([
    [0.644, 0.717, 0.267],  # Azure B
    [0.093, 0.954, 0.283],  # Eosin Y
])
S /= np.linalg.norm(S, axis=1, keepdims=True)

# Each dye's visible color (white light minus its absorption), used to color its map
DYE_CMAPS = [LinearSegmentedColormap.from_list(name, ['white', color])
             for name, color in zip(['Azure B', 'Eosin Y'], 10.0 ** (-S))]


def separate_stains(rgb, i0):
    """Beer-Lambert color deconvolution: an RGB image -> per-dye concentration maps."""
    od = -np.log10(np.clip(rgb / (i0 + 1e-6), 1e-6, 1.0))
    return np.maximum(od @ np.linalg.pinv(S), 0)

In [ ]:
demo_stem = 'IMG01117'
i0 = slides.loc[slides.slide == demo_stem, ['bg_r', 'bg_g', 'bg_b']].values[0]
crop_paths = sorted((CELLS_DIR / 'crops').glob(f'{demo_stem}_*.jpg'))

fig, axes = plt.subplots(len(crop_paths), 3, figsize=(5, 1.8 * len(crop_paths)))
axes = axes.reshape(len(crop_paths), 3)
fig.suptitle(f'Color deconvolution — {demo_stem}   I0 = {tuple(int(v) for v in i0)}')

for (ax_cell, ax_a, ax_e), crop_path in zip(axes, crop_paths):
    crop = np.array(Image.open(crop_path).convert('RGB'), dtype=float)
    mask = load_mask(CELLS_DIR / 'masks' / f'{crop_path.stem}.png')
    cell = crop.copy()
    cell[~mask] = 255  # background removed in Part 4

    concentrations = separate_stains(cell, i0)

    ax_cell.imshow(cell.astype(np.uint8), interpolation='nearest')
    ax_a.imshow(concentrations[..., 0], cmap=DYE_CMAPS[0], interpolation='nearest')
    ax_e.imshow(concentrations[..., 1], cmap=DYE_CMAPS[1], interpolation='nearest')
    for ax in (ax_cell, ax_a, ax_e):
        ax.axis('off')

for ax, title in zip(axes[0], ['cell (bg removed)', 'Azure B', 'Eosin Y']):
    ax.set_title(title, fontsize=9)

plt.tight_layout()
plt.show()

---
## Part 6 — The Precomputed Stain Maps

The concentration maps are precomputed for all ~1600 cells and saved to disk; from here the model reads them instead of deconvolving live.

**Files.** One per cell in `data/plasma_cells/stains/`, named `<slide>_<i>.tif` (same stem as its crop and mask). Each is a 3-page TIFF:

| page | content | shape | use |
|------|---------|-------|-----|
| 0 | darkfield render (RGB) | `(H, W, 3)` uint8 | preview thumbnail |
| 1 | **concentration map** | `(H, W, 2)` float | **training data** |
| 2 | stain matrix `S` | `(2, 3)` float | reproducibility |

We use **page 1** — the two dye channels from Part 5 — read with `tifffile.imread(path, key=1)`.

**Split.** `data/plasma_cells/stain_cells.csv`:
- `cell` — the stem (e.g. `IMG00083_4`); the map is `stains/<cell>.tif`
- `split` — `train` or `val`

Below we load the split and read one map back.

In [ ]:
import tifffile

STAINS_DIR = CELLS_DIR / 'stains'


def load_stain_map(cell):
    """Read the 2-channel concentration map (page 1) of a precomputed stain TIFF."""
    return tifffile.imread(STAINS_DIR / f'{cell}.tif', key=1)


stain_split = pd.read_csv(CELLS_DIR / 'stain_cells.csv')
train_cells = stain_split.loc[stain_split.split == 'train', 'cell'].tolist()
val_cells = stain_split.loc[stain_split.split == 'val', 'cell'].tolist()
print(f'{len(train_cells)} train / {len(val_cells)} val stain maps')

# Read one map back and show its two channels, as in Part 5
stain = load_stain_map(train_cells[0])
print('map shape:', stain.shape)

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
fig.suptitle(f'Precomputed map — {train_cells[0]}')
for ax, ch, name in zip(axes, range(2), ['Azure B', 'Eosin Y']):
    ax.imshow(stain[..., ch], cmap=DYE_CMAPS[ch], interpolation='nearest')
    ax.set_title(name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 7 — Train the Generative Model (Diffusion)

We use a **diffusion model** (DDPM) to generate new plasma cells. It has two directions:

- **Forward** — add Gaussian noise to a real concentration map, step by step, until it is pure noise.
- **Reverse** — a **UNet** predicts the noise added at a given step; generation starts from pure noise and removes it step by step until a clean map remains.

Training is the forward process as supervision: pick a random step, add that much noise, and train the UNet to predict the noise. Loss is MSE between predicted and true noise.

The model trains on the concentration maps from Part 6, resized to 64×64 and normalized with the dataset's per-channel `mean`/`std`.

> A usable model needs many epochs on a **GPU**. The training below is only a **demonstration** — a few epochs on a small subset, to run the loop end to end. Part 8 loads pretrained weights instead.

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import UNet2DModel, DDPMScheduler

# Per-channel normalization statistics of the concentration maps (Azure B, Eosin Y)
MEAN = torch.tensor([0.2236, 0.0584]).view(2, 1, 1)
STD = torch.tensor([0.1787, 0.0944]).view(2, 1, 1)


class StainMapDataset(Dataset):
    """2-channel stain maps, resized to a fixed size and normalized for the diffusion model."""
    def __init__(self, cells, size=64):
        self.cells, self.size = cells, size

    def __len__(self):
        return len(self.cells)

    def __getitem__(self, i):
        m = torch.from_numpy(load_stain_map(self.cells[i])).float().permute(2, 0, 1)  # (2, H, W)
        m = F.interpolate(m[None], size=self.size, mode='bilinear', align_corners=False)[0]
        return (m - MEAN) / STD


def UNet(sample_size=64, in_channels=2, out_channels=2):
    return UNet2DModel(
        sample_size=sample_size, in_channels=in_channels, out_channels=out_channels,
        block_out_channels=(64, 128, 128, 256), layers_per_block=2,
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
        up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    )


# This run only demonstrates the loop, so we train on a small subset (fast on CPU too)
demo_cells = train_cells[:64]
loader = DataLoader(StainMapDataset(demo_cells), batch_size=16, shuffle=True)
model = UNet()
scheduler = DDPMScheduler(num_train_timesteps=1000)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters, '
      f'{len(demo_cells)} maps, {len(loader)} batches/epoch')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.train()
for epoch in range(3):
    losses = []
    for images in loader:
        images = images.to(device)
        noise = torch.randn_like(images)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (images.shape[0],), device=device)
        noisy = scheduler.add_noise(images, noise, t)                 # forward process

        pred = model(noisy, t).sample                                 # UNet predicts the noise
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f'epoch {epoch + 1}/3   loss {sum(losses) / len(losses):.4f}')

---
## Part 8 — Generate Synthetic Cells

The 3-epoch model above is not enough to generate realistic cells, so we load pretrained weights from `data/models/ddpm.ckpt` (the same UNet, trained for many epochs on a GPU).

The checkpoint is a PyTorch Lightning `.ckpt`: besides the UNet it holds optimizer state, epoch counter, etc. We need only the UNet weights, which sit under the prefix `model._orig_mod.` in `state_dict` (`model.` = the UNet attribute of the Lightning module; `_orig_mod.` = added by `torch.compile`). We rebuild the same `UNet` and load them, stripping the prefix — no Lightning needed.

Generation runs the **reverse diffusion**: from Gaussian noise, remove noise over the scheduler steps down to a clean 2-channel map, then undo the normalization (`x * std + mean`).

The result is still in concentration space. To view it as RGB we invert the Part 5 deconvolution:

$$I = I_0 \cdot 10^{-\,C\cdot S}$$

for a background light $I_0$. Here we use a **near-white light** to show each cell on a pale field. (Part 9 reuses this with each template's background as $I_0$.)

In [ ]:
MODELS_DIR = Path('data/models')

# Load the pretrained UNet weights out of the Lightning checkpoint (no Lightning needed).
# weights_only=False: the checkpoint also stores config objects, and we trust this file.
ckpt = torch.load(MODELS_DIR / 'ddpm.ckpt', map_location=device, weights_only=False)
state_dict = ckpt['state_dict']
unet_weights = {k.removeprefix('model._orig_mod.'): v
                for k, v in state_dict.items() if k.startswith('model._orig_mod.')}

model = UNet().to(device)
model.load_state_dict(unet_weights)
model.eval()
print(f'Loaded {len(unet_weights)} pretrained tensors')

In [ ]:
@torch.no_grad()
def generate(n, steps=50, size=64, seed=0):
    """Sample n concentration maps by reversing the diffusion process from pure noise."""
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 2, size, size, generator=generator, device=device)

    scheduler.set_timesteps(steps)
    for t in scheduler.timesteps:
        noise_pred = model(x, t).sample
        x = scheduler.step(noise_pred, t, x).prev_sample

    x = x * STD.to(device) + MEAN.to(device)          # undo the normalization
    return x.clamp(min=0).cpu().permute(0, 2, 3, 1)   # (n, H, W, 2)


WHITE_LIGHT = np.array([245, 245, 245])  # near-white light, just for viewing the cells


def stains_to_rgb(concentration, i0):
    """Inverse of the Part 5 deconvolution: rebuild RGB from concentrations under a light i0."""
    return (i0 * 10.0 ** (-(concentration @ S))).clip(0, 255).astype(np.uint8)


samples = generate(6)

fig, axes = plt.subplots(1, len(samples), figsize=(1.7 * len(samples), 2.2))
fig.suptitle('Generated synthetic cells (RGB, near-white light)')
for ax, cell in zip(axes, samples):
    ax.imshow(stains_to_rgb(cell.numpy(), WHITE_LIGHT), interpolation='nearest')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 9 — Compose Full Synthetic Smears

We combine the two halves of the pipeline:

- a **template** (Part 3): a background color and non-overlapping `(x, y, size)` boxes;
- the **generative model** (Part 8): one plasma cell per box.

Each generated concentration map becomes a colored cell via `stains_to_rgb` from Part 8 (inverse Beer-Lambert), now with the **template's background color** as $I_0$.

The model's background is not exactly zero, so a raw square paste leaves a faint tinted box. We build a **soft mask** from the concentration — opaque on the cell, transparent where it is near zero — and paste through it, so only the cell lands and the template background shows elsewhere.

Per box: generate a cell, rebuild its RGB, paste through its mask. The result is a full synthetic smear.

In [ ]:
def compose_smear(bg, boxes, canvas_size=512, seed=0):
    """Paste one generated cell into each template box, rebuilt as RGB under the template's light."""
    canvas = Image.new('RGB', (canvas_size, canvas_size), tuple(int(v) for v in bg))
    for (x, y, s), cell in zip(boxes, generate(len(boxes), seed=seed)):
        cell = cell.numpy()
        patch = Image.fromarray(stains_to_rgb(cell, bg))
        # Soft alpha from concentration: opaque on the cell, transparent on its near-zero background
        alpha = np.clip((cell.sum(-1) - 0.05) / 0.15, 0, 1)
        mask = Image.fromarray((alpha * 255).astype(np.uint8))

        side = max(1, round(s * canvas_size))
        patch, mask = patch.resize((side, side)), mask.resize((side, side))
        canvas.paste(patch, (round(x * canvas_size) - side // 2, round(y * canvas_size) - side // 2), mask)
    return canvas


rng = np.random.default_rng(0)  # reproducible templates

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Synthetic bone marrow smears')
for i, ax in enumerate(axes):
    bg, boxes = sample_template(rng)
    ax.imshow(compose_smear(bg, boxes, seed=i))
    ax.set_title(f'{len(boxes)} cells   RGB {tuple(int(v) for v in bg)}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 10 — Test-Set Detection: Baseline vs. Augmented

We compare two YOLO26-n detectors: `data/models/yolo26n_baseline.pt` (trained on the real training set) and `data/models/yolo26n_aug.pt` (same data plus 40% DDPM-synthesized plasma cells from this pipeline). Both predict a single class, `plasma_cell`.

Each runs at confidence 0.25. Predictions (green, dashed) overlay the ground-truth boxes (red; class 0 only). Two test images where the augmented model fixes a baseline failure:

- `P_20221113_162935.jpg` — baseline emits four false positives; aug returns the three true cells only.
- `P_20221113_165335.jpg` — baseline misses one of three cells; aug detects all three, with no false positives in either.

In [ ]:
from ultralytics import YOLO
from matplotlib.patches import Rectangle

baseline = YOLO(MODELS_DIR / 'yolo26n_baseline.pt')
aug = YOLO(MODELS_DIR / 'yolo26n_aug.pt')

TEST_DIR = SLIDES_DIR / 'test'
examples = ['patient_02/images/P_20221113_162935.jpg',
            'patient_05/images/P_20221113_165335.jpg']


def gt_boxes(img_path, w, h):
    """Ground-truth plasma_cell (class 0) boxes, in pixel xyxy."""
    label_path = img_path.parents[1] / 'labels' / f'{img_path.stem}.txt'
    boxes = []
    for c, x, y, bw, bh in np.loadtxt(label_path, ndmin=2):
        if int(c) == 0:
            boxes.append([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h])
    return boxes


def draw(ax, img, gt, pred, title):
    ax.imshow(img)
    for b in gt:  # ground truth
        ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor='red', lw=2))
    for b in pred:  # detections
        ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor='mediumseagreen', lw=1.5, ls='--'))
    ax.set_title(title, fontsize=9)
    ax.axis('off')


fig, axes = plt.subplots(len(examples), 2, figsize=(10, 5 * len(examples)))
fig.suptitle('Test-set detections — GT (red), prediction (green dashed)')
for row, rel in zip(axes, examples):
    img_path = TEST_DIR / rel
    img = Image.open(img_path).convert('RGB')
    gt = gt_boxes(img_path, *img.size)
    for ax, det, name in zip(row, [baseline, aug], ['baseline', 'aug']):
        pred = det.predict(img_path, conf=0.25, verbose=False)[0].boxes.xyxy.cpu().numpy()
        draw(ax, img, gt, pred, f'{img_path.name} — {name}: {len(pred)} preds')
plt.tight_layout()
plt.show()